**NN TFLITE TEMPLATE**

*This notebook will pull in the class data that has been collated for sitting, standing, walking, and falling. It provides a basis for creating an ML model that will take in gyroscope/accelerometer data from the Raspberry Pi and determine what behavior is being performed.*

**Part I. Set-Up**

First, we need to make sure we are working with a version of Tensorflow that complements our capabilities on our embedded system.

The Pi runs Python 3.7. We will upgrade to use a tflite_runtime interpreter == 2.11.0 BUT we will still have to downgrade tensorflow (for model development) to 2.12.0 so that the oeprations are compatible for inference on our embedded system.

In [1]:
!pip uninstall tensorflow
!pip install tensorflow==2.12.0

Found existing installation: tensorflow 2.12.0
Uninstalling tensorflow-2.12.0:
  Would remove:
    /usr/local/bin/estimator_ckpt_converter
    /usr/local/bin/import_pb_to_tensorboard
    /usr/local/bin/saved_model_cli
    /usr/local/bin/tensorboard
    /usr/local/bin/tf_upgrade_v2
    /usr/local/bin/tflite_convert
    /usr/local/bin/toco
    /usr/local/bin/toco_from_protos
    /usr/local/lib/python3.11/dist-packages/tensorflow-2.12.0.dist-info/*
    /usr/local/lib/python3.11/dist-packages/tensorflow/*
Proceed (Y/n)? Y
  Successfully uninstalled tensorflow-2.12.0
  Using cached tensorflow-2.12.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.4 kB)
Using cached tensorflow-2.12.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (586.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.

In [2]:
#Make sure it is 2.12.0 before doing any work
import tensorflow as tf
print(tf.__version__)

2.12.0


Next we need to mount our Drive and map to where our data lives. Please make sure that the CSV files given in the github repository are saved in a folder of your choosing on your Google Drive. Then map to that directory. Feel free to use

```
!ls
```

 to check that your files are there!

In [4]:
# prompt: navigate to a folder in my google drive My Drive > data > fall_data

import os
from google.colab import drive

drive.mount('/content/drive')
%cd /content/drive/My Drive/Data/Fall Data
!pwd


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/Data/Fall Data
/content/drive/My Drive/Data/Fall Data


In [5]:
!ls

falling_data.csv	   sitting_data_windowed.csv
falling_data_windowed.csv  standing_data.csv
nn_model_raw.tflite	   standing_data_windowed.csv
rf_model.pkl		   walking_data.csv
sitting_data.csv	   walking_data_windowed.csv


**Part II. Data Exploration**

Now let's pull in the data using


```
pd.read_csv()
```
Label the columns and explore this data set.


*   How many data points are there in each?
*   Maybe find a way to plot a couple of the parameters over time to see what the "behavior" looks like?




In [6]:
import pandas as pd

# Load the four CSV files into dataframes without the index
df_sitting = pd.read_csv('sitting_data.csv', index_col=False)
df_standing = pd.read_csv('standing_data.csv', index_col=False)
df_walking = pd.read_csv('walking_data.csv', index_col=False)
df_falling = pd.read_csv('falling_data.csv', index_col=False)

# Specify the columns that contain gyro and accel data
gyro_accel_columns = ['Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Accel_X', 'Accel_Y', 'Accel_Z']  # Replace with actual column names



In [7]:
print("Data points:")
print("Sitting:", len(df_sitting))
print("Standing:", len(df_standing))
print("Walking:", len(df_walking))
print("Falling:", len(df_falling))

print("\nSample of 'walking' data:")
print(df_walking.head())

print("\nMissing values:")
print(pd.concat([df_sitting, df_standing, df_walking, df_falling]).isnull().sum())

combined_df = pd.concat([df_sitting, df_standing, df_walking, df_falling], ignore_index=True)

print("\nSummary:")
print(combined_df[['Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Accel_X', 'Accel_Y', 'Accel_Z']].describe())

print("\nUnique labels:")
print(combined_df['Label'].value_counts())


Data points:
Sitting: 2800
Standing: 2800
Walking: 2800
Falling: 2800

Sample of 'walking' data:
   Index    Gyro_X    Gyro_Y    Gyro_Z   Accel_X   Accel_Y   Accel_Z  Label
0      1  0.199229  0.239901 -0.895492  0.001212  0.114598  1.048871      2
1      2 -0.020987  0.095432 -0.797448  0.120253 -0.123508  0.995014      2
2      3 -0.269304 -0.038042 -0.715286  0.111525 -0.096973  0.918250      2
3      4  0.052622  0.065805 -0.851815  0.082917 -0.129538  0.978687      2
4      5 -0.243342  0.027015 -1.007280  0.071522 -0.135087  1.006224      2

Missing values:
Index      0
Gyro_X     0
Gyro_Y     0
Gyro_Z     0
Accel_X    0
Accel_Y    0
Accel_Z    0
Label      0
dtype: int64

Summary:
             Gyro_X        Gyro_Y        Gyro_Z       Accel_X       Accel_Y  \
count  11200.000000  11200.000000  11200.000000  11200.000000  11200.000000   
mean       0.005709     -0.005832     -0.036415      0.052052      0.061181   
std        0.547987      0.629744      0.639754      0.215303     

**Part III. Tensorflow**

Now let's use NN. We will start with a simple NN that takes in raw data.

We also have to think about what libraries will be using and what versions are available for our Raspberry Pi.

In [8]:
# Import libraries (add libraries you need here)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from sklearn import svm
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

You may want to concatenate all of your dataframes into one dataframe.

In [11]:
# Combine
X = pd.concat([df_sitting, df_standing, df_walking, df_falling], ignore_index=True)
X = shuffle(X, random_state=42)

# Drop the index column and separate features/labels
X = X.drop('Index', axis=1)
y = X['Label'].values
X = X.drop('Label', axis=1)

# Encode the labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Convert arrays to NumPy
X_array = X.to_numpy()
y_array = np.array(y_encoded)

# Train and test splt
X_train, X_test, y_train, y_test = train_test_split(X_array, y_array, test_size=0.2, random_state=42)

# Reshape input
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# CNN
model = Sequential()
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))
model.add(Flatten())
model.add(Dense(100, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(4, activation='softmax'))  # 4 output classes

# Compile and train
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=25, batch_size=32, validation_split=0.2)

# Evaluate
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Optimized CNN accuracy: {accuracy:.2f}")


Epoch 1/25
224/224 [==============================] - 1s 3ms/step - loss: 1.1178 - accuracy: 0.5066 - val_loss: 0.9717 - val_accuracy: 0.6122
Epoch 2/25
224/224 [==============================] - 0s 2ms/step - loss: 0.9747 - accuracy: 0.5844 - val_loss: 0.9056 - val_accuracy: 0.6373
Epoch 3/25
224/224 [==============================] - 0s 2ms/step - loss: 0.9201 - accuracy: 0.6000 - val_loss: 0.8723 - val_accuracy: 0.6289
Epoch 4/25
224/224 [==============================] - 0s 2ms/step - loss: 0.8938 - accuracy: 0.6169 - val_loss: 0.8432 - val_accuracy: 0.6462
Epoch 5/25
224/224 [==============================] - 0s 2ms/step - loss: 0.8711 - accuracy: 0.6242 - val_loss: 0.8359 - val_accuracy: 0.6367
Epoch 6/25
224/224 [==============================] - 1s 2ms/step - loss: 0.8643 - accuracy: 0.6249 - val_loss: 0.8246 - val_accuracy: 0.6496
Epoch 7/25
224/224 [==============================] - 0s 2ms/step - loss: 0.8624 - accuracy: 0.6302 - val_loss: 0.8088 - val_accuracy: 0.6641
Epoch 

In [12]:
import tensorflow as tf
#Set-up
model = model
model_file_name = 'nn_model_raw_1np.tflite'
#Convert the model to TensorFlow Lite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

#Save the TensorFlow Lite model to a file
with open(model_file_name, 'wb') as f:
    f.write(tflite_model)

print("Model converted to TensorFlow Lite and saved as ", model_file_name)

Model converted to TensorFlow Lite and saved as  nn_model_raw_1np.tflite


**Part IV. Deploy Model on Embedded Device**

In order to deploy the model, the model file needs to be saved and a virtual environment should be created that installs all necessary libraries.

Below is a script that pulls data from the SenseHat, consolidates it into input for the model, makes a prediction and changes the color of the LED Matrix based on the prediction.

This file should be saved on the Pi as  something like predict_nn_1np.py
nn = neural network
1np = 1 numpy array (1 x 6 features input).

Also remember that the data input into the model should match how you did it during training!



In [ ]:
# ON Raspberry Pi Script! nn_model_raw_1np.tflite

# This script should be saved as nn_predict.py on your device.
# It should be in the same directory as your nn_model_raw_1np.tflite file.
# YOU NEED A PYTHON ENVIRONMENT SET UP BEFORE RUNNING THIS!

# This assumes a 1 x 6 feature numpy array as an input for the model.

import numpy as np
from sense_hat import SenseHat
import tflite_runtime.interpreter as tflite

# Initialize SenseHat
sense = SenseHat()

# Load the TensorFlow Lite model
interpreter = tflite.Interpreter(model_path="nn_model_raw_1np.tflite")
interpreter.allocate_tensors()

# Get input and output tensors
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Function to get gyro/accel data from SenseHat
def get_gyro_accel_data():
    accel = sense.get_accelerometer_raw()
    gyro = sense.get_gyroscope_raw()
    data = np.array([accel['x'], accel['y'], accel['z'], gyro['x'], gyro['y'], gyro['z']])
    return data

# Function to make a prediction based on gyro/accel data
def predict(data):
    # Preprocess the data to match the input shape of the model
    data = data.reshape((1,6)).astype(np.float32)

    # Set the tensor to point to the input data to be inferred
    interpreter.set_tensor(input_details[0]['index'], data)

    # Run inference
    interpreter.invoke()

    # Get the prediction results
    output_data = interpreter.get_tensor(output_details[0]['index'])
    prediction = np.argmax(output_data)

    return prediction

# Function to update LED matrix based on prediction
def update_led_matrix(prediction):
    if prediction == 0:
        color = (0, 0, 255)  # Blue for sitting
        print("Sitting")
    elif prediction == 1:
        color = (255, 255, 0)  # Yellow for standing
        print("Standing")
    elif prediction == 2:
        color = (0, 255, 0)  # Green for walking
        print("Walking")
    elif prediction == 3:
        color = (255, 0, 0)  # Red for falling
        print("Falling")

    # Update LED matrix with the color
    sense.clear(color)

# Main loop to continuously get data, make predictions, and update LED matrix

while True:
    data = get_gyro_accel_data()
    prediction = predict(data)
    update_led_matrix(prediction)